In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%sql
USE CATALOG databricksdemo;
USE SCHEMA silver;

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@storageaccountsunnybest.dfs.core.windows.net/orders")

In [0]:
display(df)

In [0]:
df.withColumnRenamed("_rescued_data", "rescued_data").display()


In [0]:
df = df.drop("_rescued_data")\
    .withColumn('order_date', to_timestamp('order_date'))\
    .withColumn("year", year('order_date'))

display(df)


In [0]:
window_spec = Window.partitionBy("year").orderBy(desc("total_amount"))

df.withColumn("rank", rank().over(window_spec))\
    .withColumn('dense_rank', dense_rank().over(window_spec))\
    .withColumn("row_number", row_number().over(window_spec))\
    .display()

### Classes

In [0]:
class Windows:
    def __init__(self, df):
        self.df = df
        self.window_spec = (
            Window.partitionBy("year")
            .orderBy(col("total_amount").desc())
        )

    def dense_rnk(self):
        return self.df.withColumn("dense_rank", dense_rank().over(self.window_spec))

    def rnk(self):
        return self.df.withColumn("rank", rank().over(self.window_spec))

    def row_num(self):
        return self.df.withColumn("row_number", row_number().over(self.window_spec))

In [0]:
display(Windows(df).dense_rnk())

## Data Writing

In [0]:
display(df)

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("abfss://silver@storageaccountsunnybest.dfs.core.windows.net/orders")


In [0]:
%sql
CREATE TABLE IF NOT EXISTS orders_silver
USING delta
LOCATION 'abfss://silver@storageaccountsunnybest.dfs.core.windows.net/orders'

In [0]:
%sql
SELECT *
FROM orders_silver;